python -m venv venv
venv\Scripts\activate
pip install numpy matplotlib tensorflow scikit-learn
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
pip install notebook
python -m notebook

In [71]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [72]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = datasets.MNIST(root='mnist_data', train=True, download=True, transform=transform)
testset = datasets.MNIST(root='mnist_data', train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=100, shuffle=False)

print(f'Training samples: {len(trainset)}')
print(f'Test samples: {len(testset)}')

100.0%
100.0%
100.0%
100.0%


Training samples: 60000
Test samples: 10000


In [73]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(1600, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = nn.functional.relu(self.conv1(x))
        x = nn.functional.max_pool2d(x, 2)
        x = nn.functional.relu(self.conv2(x))
        x = nn.functional.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return nn.functional.log_softmax(x, dim=1)

pt_model = SimpleCNN()
print(pt_model)

SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=1600, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [74]:
criterion = nn.NLLLoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.001)

for epoch in range(2):
    pt_model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in trainloader:
        optimizer.zero_grad()
        output = pt_model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(output, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}, Accuracy: {correct/total:.4f}")

Epoch 1, Accuracy: 0.9508
Epoch 2, Accuracy: 0.9855


In [75]:
pt_model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in testloader:
        output = pt_model(images)
        _, predicted = torch.max(output, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

pt_accuracy = correct / total
print(f'PyTorch Test Accuracy: {pt_accuracy:.4f}')

PyTorch Test Accuracy: 0.9879


In [76]:
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(-1, 28, 28, 1) / 255.0
x_test = x_test.reshape(-1, 28, 28, 1) / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [77]:
from tensorflow.keras import layers, models

k_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

C:\A-Projects-gng\practicals\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [78]:
k_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

k_model.fit(x_train, y_train, epochs=2, batch_size=64, validation_split=0.1)

k_loss, k_accuracy = k_model.evaluate(x_test, y_test, verbose=0)
print(f'Keras Test Accuracy: {k_accuracy:.4f}')

Epoch 1/2
844/844 ━━━━━━━━━━━━━━━━━━━━ 24s 25ms/step - accuracy: 0.9382 - loss: 0.2024 - val_accuracy: 0.9817 - val_loss: 0.0676
Epoch 2/2
844/844 ━━━━━━━━━━━━━━━━━━━━ 20s 24ms/step - accuracy: 0.9805 - loss: 0.0620 - val_accuracy: 0.9828 - val_loss: 0.0546
Keras Test Accuracy: 0.9820


In [80]:
import tensorflow as tf
from tensorflow.keras import layers, models

#Ensure labels are correct dtype
y_train = tf.cast(y_train, tf.int32)

tf_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)) \
                          .shuffle(10000) \
                          .batch(64)

for epoch in range(2):
    correct, total = 0, 0

    for x_batch, y_batch in train_ds:
        with tf.GradientTape() as tape:
            preds = tf_model(x_batch, training=True)
            loss = loss_fn(y_batch, preds)

        grads = tape.gradient(loss, tf_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, tf_model.trainable_variables))

        # Fix: match dtype explicitly
        pred_labels = tf.argmax(preds, axis=1, output_type=tf.int32)

        correct += tf.reduce_sum(
            tf.cast(pred_labels == y_batch, tf.int32)
        ).numpy()

        total += y_batch.shape[0]

    print(f"Epoch {epoch+1}, Accuracy: {correct/total:.4f}")

Epoch 1, Accuracy: 0.9486
Epoch 2, Accuracy: 0.9837


In [82]:
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(100)

correct, total = 0, 0
for x_batch, y_batch in test_ds:
    preds = tf_model(x_batch, training=False)

    # Fix dtype here
    pred_labels = tf.argmax(preds, axis=1, output_type=tf.int32)
    y_batch = tf.cast(y_batch, tf.int32)

    correct += tf.reduce_sum(
        tf.cast(pred_labels == y_batch, tf.int32)
    ).numpy()

    total += y_batch.shape[0]

tf_accuracy = correct / total
print(f'TensorFlow Test Accuracy: {tf_accuracy:.4f}')

TensorFlow Test Accuracy: 0.9849


In [83]:
print("="*40)
print("Framework Comparison on MNIST")
print("="*40)
print(f"PyTorch Accuracy: {pt_accuracy:.4f}")
print(f"Keras Accuracy: {k_accuracy:.4f}")
print(f"TensorFlow Accuracy: {tf_accuracy:.4f}")
print("="*40)

Framework Comparison on MNIST
PyTorch Accuracy: 0.9879
Keras Accuracy: 0.9820
TensorFlow Accuracy: 0.9849
